In [1]:
import numpy as np
import random
import matplotlib.pyplot as plt

In [2]:
class child:
    def __init__(self, gain_capability=-1):
        # there will also be an individual personality matrix for each child that will be randomized
        self.emotion_rigidity = random.uniform(.1, .3) # how easy it will be to change emotions
        self.distraction_susceptibility = random.uniform(0.0, 0.4) # how easily can the child get distracted
        
        if gain_capability == -1:
            self.current_gain_capability = random.randint(0, 4)
        else: 
            self.current_gain_capability = gain_capability
            
        # the state vars for the child-
        self.emotion = 0 # discrete b/w 0-3 (0: neutral, 1: angry, 2: happy, 3: agitated/afraid/anxious)
        self.distraction = 0 # discrete ordinal between 0,1,2,3,4
        self.current_difficulty = 0 # 5 levels, increasingly tough. 0 through 4
        self.knowledge_gain = 0 # postive reward
    
    def return_state(self):
        # Returns exactly 4 dimensions to match the Q-Table state space. 
        # (knowledge_gain is a reward, not a persistent state, so it is removed here).
        return (
            min(max(self.emotion, 0), 3),
            min(max(self.distraction, 0), 4),
            min(max(self.current_difficulty, 0), 4),
            min(max(self.current_gain_capability, 0), 4)
        )

    def step(self, action):
        # FIXED: Standard RL architecture. The environment handles the transition and returns feedback.
        self.take_model_suggestion(action)
        self.elapse_time()
        reward = self._calc_reward()
        next_state = self.return_state()
        
        return next_state, reward

    def _calc_reward(self):
        # Moved from model. The environment defines its own reward logic.
        reward = 0
        reward += self.knowledge_gain * 2
        reward -= self.distraction * 1

        if self.emotion == 0:   # neutral
            reward += 1
        elif self.emotion == 1: # angry
            reward -= 2
        elif self.emotion == 2: # happy
            reward += 2
        elif self.emotion == 3: # anxious/agitated
            reward -= 3
            
        return reward

    def take_model_suggestion(self, action):
        '''
        action vals can be 0 "music", 1 "animations", 2 "decrease difficulty", 3 "increase difficulty", 4 "nothing"
        '''
        if action == 0:
            self.simulate_music()
        elif action == 1:
            self.simulate_animation()
        elif action == 2:
            self.simulate_dec_difficulty()
        elif action == 3:
            self.simulate_inc_difficulty()
        elif action == 4:
            pass #do nothing action
        
    def simulate_music(self): 
        if random.random() >= self.emotion_rigidity:
            if self.emotion == 0: self.emotion = 2
            else: self.emotion = 0
        if random.random() < self.distraction_susceptibility: 
            self.distraction = min(4, self.distraction + 1)

    def simulate_animation(self): 
        self.distraction = 0
        if random.random() < self.distraction_susceptibility: 
            self.distraction += 1

    def simulate_dec_difficulty(self):
        # FIXED: Changed -1 to 0. Negative indexing in Python would wrap around to index 4 in the Q-table.
        self.current_difficulty = max(0, self.current_difficulty - 1) 
            
    def simulate_inc_difficulty(self):
        self.current_difficulty = min(4, self.current_difficulty + 1)

    def elapse_time(self):
        # FIXED: Reset knowledge gain at the start of the time step, so the model doesn't have to mutate it.
        self.knowledge_gain = 0 
        
        if self.current_difficulty <= self.current_gain_capability and self.distraction < 2:
            self.knowledge_gain = min(4, self.current_difficulty + 1)
        else: 
            if random.random() >= .75:  
                self.knowledge_gain = self.current_difficulty
            else:
                self.distraction = min(self.distraction + 1, 4)
                if self.emotion_rigidity > random.random(): 
                    self.emotion = 3
        
        if random.random() < self.distraction_susceptibility and self.distraction != 0:
            self.distraction = min(self.distraction + 1, 4)
            
        if random.random() < self.emotion_rigidity:
            self.emotion = random.randint(0, 3)




In [3]:
class DecisionModel:
    def __init__(self, childObj: child, modelname: str):
        self.q_table = np.zeros(shape=(4, 5, 5, 5, 5)) # (emotion, distraction, difficulty, gain_capability, action)
        self.alpha = 0.1  # learning rate
        self.discount_factor = 0.9
        self.actions = [0, 1, 2, 3, 4] # music, animation, dec diff, inc diff, nothing   
        self.random_enable = True 
        self.epsilon = 0.4 
        self.epsilon_decay = 0.99995 
        self.childObj = childObj
        self.name = modelname
        self.reward_log = []
        self.action_log = []
        self.epsilon_log = []
    def resetlearningParameters(self):
        self.epsilon = 0.4

    def choose_and_execute_action(self):
        cur_state = self.childObj.return_state()
        
        # choose action
        if random.random() < self.epsilon and self.random_enable:
            action = random.choice(self.actions)
        else:
            # Cleanly unpack the specific state tuple to get the array of 5 action values
            action = np.argmax(self.q_table[cur_state])
            
        # execute action and get feedback purely through the step() method
        next_state, reward = self.childObj.step(action)
        
        self.updateQtable(cur_state, action, reward, next_state)

        # log
        self.reward_log.append(reward)
        self.action_log.append(action)
        self.epsilon_log.append(self.epsilon)
        
        return f"State: {cur_state}, Action: {action}, Reward: {reward}, Next: {next_state}"
    
    def updateQtable(self, cur_state, action, reward, next_state):
        # Directly index using the tuple to avoid dimension mismatch
        current_q = self.q_table[cur_state][action]
        max_future_q = np.max(self.q_table[next_state])
        
        new_q = current_q + self.alpha * (reward + self.discount_factor * max_future_q - current_q)
        self.q_table[cur_state][action] = new_q

    def run(self, iterations, iteration_print: bool = True):
        for i in range(iterations):
            if iteration_print:
                print(f"{self.name} Iteration: {i} ", end="")
            
            statement = self.choose_and_execute_action()
            
            if iteration_print: 
                print(statement)
                
            # self.childObj.knowledge_gain=0 
            self.epsilon *= self.epsilon_decay
            
    def saveModel(self):
        np.save(f"Qtable{self.name}.npy", self.q_table)
    
    def plot_learning_curve(self, window=100):
        rewards = np.array(self.reward_log)
        moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')

        plt.figure()
        plt.plot(moving_avg)
        plt.xlabel("Iterations")
        plt.ylabel("Average Reward")
        plt.title(f"Learning Curve - {self.name}")
        plt.grid(True)
        plt.show()
        
    def plot_action_distribution(self, window=200):
        actions = np.array(self.action_log)

        plt.figure()
        for a in self.actions:
            freq = np.convolve(
                (actions == a).astype(int),
                np.ones(window)/window,
                mode='valid'
            )
            plt.plot(freq, label=f"Action {a}")

        plt.xlabel("Iterations")
        plt.ylabel("Frequency")
        plt.title(f"Action Selection Over Time - {self.name}")
        plt.legend()
        plt.grid(True)
        plt.show()
        
    def plot_epsilon_decay(self):
        plt.figure()
        plt.plot(self.epsilon_log)
        plt.xlabel("Iterations")
        plt.ylabel("Epsilon")
        plt.title(f"Exploration Decay - {self.name}")
        plt.grid(True)
        plt.show()